Pasos a seguir para la ejecución del modelo en local:
1. Crear entorno XXX con python 3.9
- conda create -n XXX python=3.9
2. Instalación de librerías con estas versiones para evitar incompatibilidades
- pip install "tensorflow==2.9.3"
- pip install numpy==1.24.4
- pip install opencv-python==4.8.0.76
3. Descargar el zip de la web de Teachable Machine y extraer los ficheros keras_model.h5 y labels.txt en el mismo directorio que este notebook
4. [EXTRA] "Jugar" con alguna LLM para customizar la salida ventana de imágenes

In [ ]:
# Código propuesto por Teachable Machine

from keras.models import load_model  # TensorFlow is required for Keras to work
import cv2  # Install opencv-python
import numpy as np

# Disable scientific notation for clarity
np.set_printoptions(suppress=True)

# Load the model
# model = load_model("keras_model_converted.keras", compile=False)
model = load_model("keras_model.h5", compile=False)

# Load the labels
class_names = open("labels.txt", "r").readlines()

# CAMERA can be 0 or 1 based on default camera of your computer
camera = cv2.VideoCapture(0)

while True:
    # Grab the webcamera's image.
    ret, image = camera.read()

    # Resize the raw image into (224-height,224-width) pixels
    image = cv2.resize(image, (224, 224), interpolation=cv2.INTER_AREA)

    # Show the image in a window
    cv2.imshow("Webcam Image", image)

    # Make the image a numpy array and reshape it to the models input shape.
    image = np.asarray(image, dtype=np.float32).reshape(1, 224, 224, 3)

    # Normalize the image array
    image = (image / 127.5) - 1

    # Predicts the model
    prediction = model.predict(image)
    index = np.argmax(prediction)
    class_name = class_names[index]
    confidence_score = prediction[0][index]

    # Print prediction and confidence score
    print("Class:", class_name[2:], end="")
    print("Confidence Score:", str(np.round(confidence_score * 100))[:-2], "%")

    # Listen to the keyboard for presses.
    keyboard_input = cv2.waitKey(1)

    # 27 is the ASCII for the esc key on your keyboard.
    if keyboard_input == 27:
        break

camera.release()
cv2.destroyAllWindows()


1/1 [==============================] - 1s 803ms/step
Class: Botella
Confidence Score: 71 %
1/1 [==============================] - 0s 29ms/step
Class: Botella
Confidence Score: 73 %
1/1 [==============================] - 0s 33ms/step
Class: Botella
Confidence Score: 71 %
1/1 [==============================] - 0s 30ms/step
Class: Botella
Confidence Score: 71 %
1/1 [==============================] - 0s 33ms/step
Class: Botella
Confidence Score: 65 %
1/1 [==============================] - 0s 26ms/step
Class: Botella
Confidence Score: 65 %
1/1 [==============================] - 0s 22ms/step
Class: Botella
Confidence Score: 65 %
1/1 [==============================] - 0s 25ms/step
Class: Botella
Confidence Score: 65 %
1/1 [==============================] - 0s 25ms/step
Class: Botella
Confidence Score: 65 %
1/1 [==============================] - 0s 26ms/step
Class: Botella
Confidence Score: 65 %
1/1 [==============================] - 0s 36ms/step
Class: Botella
Confidence Score: 65 %
1/1 [====

In [ ]:
#Primeras modificaciones con apoyo de LLM

from keras.models import load_model  # TensorFlow is required for Keras to work
import cv2  # Install opencv-python
import numpy as np

# Disable scientific notation for clarity
np.set_printoptions(suppress=True)

# Load the model
# model = load_model("keras_model_converted.keras", compile=False)
model = load_model("keras_model.h5", compile=False)

# Load the labels
class_names = open("labels.txt", "r").readlines()

# CAMERA can be 0 or 1 based on default camera of your computer
camera = cv2.VideoCapture(0)

while True:
    # Captura la imagen
    ret, image = camera.read()
    if not ret:
        break

    # Redimensiona y preprocesa
    input_image = cv2.resize(image, (224, 224))
    input_array = (np.asarray(input_image, dtype=np.float32).reshape(1, 224, 224, 3) / 127.5) - 1

    # Predicción
    prediction = model.predict(input_array)[0]  # forma (num_clases,)
    top_indices = prediction.argsort()[-2:][::-1]  # indices de las 2 clases más probables

    # Escribe las dos etiquetas con sus porcentajes sobre la imagen
    for i, idx in enumerate(top_indices):
        label_text = f"{class_names[idx].strip()}: {prediction[idx]*100:.1f}%"
        cv2.putText(
            image,
            label_text,
            (10, 30 + i*40),      # separa cada línea 40px
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2,
            cv2.LINE_AA
        )

    # Muestra la imagen
    cv2.imshow("Webcam Image", image)

    # Salir con Esc
    if cv2.waitKey(1) == 27:
        break

camera.release()
cv2.destroyAllWindows()


1/1 [==============================] - 0s 20ms/step
